# 21cmFAST + patchy kSZ + non-Gaussianity through the EoR

Run-it-top-to-bottom notebook. Config, simulations, and non-Gaussian stats live here; the kSZ functions are in `ksz.py` and the plot helpers are in `plotting.py`.

The lightcone kSZ routine is self-contained inside `ksz.py`, so a stock `py21cmfast` install is enough &mdash; no custom `wrapper.py` swap required. Just install system deps + `21cmFAST` (see `README.md`).

## Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import logging
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

import py21cmfast as p21c
from py21cmfast import UserParams, CosmoParams, AstroParams, FlagOptions

from ksz import ksz_from_coeval, ksz_from_lightcone
from plotting import (plot_evolution, plot_pdf_evolution,
                      plot_power_spectrum, plot_ksz_map, plot_lightcone_pdf)

## 1. Config

All tunable parameters are in this cell. Defaults are small so the demo runs in minutes on a laptop — bump `BOX_LEN` and `HII_DIM` for a real run.

In [ ]:
# --- simulation box ---
BOX_LEN = 200.0        # cMpc
HII_DIM = 100          # ionisation grid / side
DIM = 3 * HII_DIM      # high-res density grid
random_seed = 12345

# --- coeval cubes (redshifts span the EoR) ---
coeval_redshifts = [15.0, 12.0, 10.0, 9.0, 8.0, 7.5, 7.0, 6.5, 6.0]

# --- lightcone range ---
z_min, z_max = 5.5, 15.0
lightcone_quantities = ('brightness_temp', 'xH_box', 'density', 'velocity')
global_quantities    = ('brightness_temp', 'xH_box')

# --- cosmology (Planck 2018) ---
hlittle, OMm, OMb = 0.6766, 0.30964, 0.04897
POWER_INDEX, SIGMA_8 = 0.9665, 0.8102
Y_He = 0.245

# --- astrophysics ---
HII_EFF_FACTOR = 30.0
ION_Tvir_MIN   = 4.7

# --- physics flags ---
USE_TS_FLUCT = False
INHOMO_RECO  = False
PHOTON_CONS  = False
USE_MASS_DEPENDENT_ZETA = True

# --- kSZ options ---
ksz_z_start = 5.5
ksz_rotation = True
ksz_parallel_approx = False

# --- non-Gaussian stats ---
pdf_bins = 80
pk_nbins = 20
pk_log_bins = True

# --- IO ---
cache_dir  = Path('./_21cmfast_cache')
output_dir = Path('./outputs')
figure_dir = Path('./figures')
for d in (cache_dir, output_dir, figure_dir):
    d.mkdir(parents=True, exist_ok=True)
p21c.config['direc'] = str(cache_dir)

# --- build the py21cmfast parameter objects ---
user_params  = UserParams(HII_DIM=HII_DIM, DIM=DIM, BOX_LEN=BOX_LEN)
cosmo_params = CosmoParams(hlittle=hlittle, OMm=OMm, OMb=OMb,
                           POWER_INDEX=POWER_INDEX, SIGMA_8=SIGMA_8)
astro_params = AstroParams(HII_EFF_FACTOR=HII_EFF_FACTOR, ION_Tvir_MIN=ION_Tvir_MIN)
flag_options = FlagOptions(USE_TS_FLUCT=USE_TS_FLUCT,
                           INHOMO_RECO=INHOMO_RECO,
                           USE_MASS_DEPENDENT_ZETA=USE_MASS_DEPENDENT_ZETA,
                           PHOTON_CONS=PHOTON_CONS)
print('config ready: BOX_LEN', BOX_LEN, 'cMpc, HII_DIM', HII_DIM)
print('coeval z:', coeval_redshifts)
print('lightcone z:', z_min, '->', z_max)

## 2. Run 21cmFAST — coeval cubes

In [ ]:
coevals = p21c.run_coeval(
    redshift=coeval_redshifts,
    user_params=user_params,
    cosmo_params=cosmo_params,
    astro_params=astro_params,
    flag_options=flag_options,
    random_seed=random_seed,
    direc=str(cache_dir),
    write=True,
)
if not isinstance(coevals, (list, tuple)):
    coevals = [coevals]
# sort high-z -> low-z so plots read early-to-late EoR
coevals = sorted(coevals, key=lambda c: -c.redshift)

for c in coevals:
    print(f'z={c.redshift:5.2f}   mean x_HI={float(np.mean(c.xH_box)):.3f}')

## 3. Run 21cmFAST — lightcone

In [ ]:
lc = p21c.run_lightcone(
    redshift=z_min,
    max_redshift=z_max,
    user_params=user_params,
    cosmo_params=cosmo_params,
    astro_params=astro_params,
    flag_options=flag_options,
    lightcone_quantities=lightcone_quantities,
    global_quantities=global_quantities,
    random_seed=random_seed,
    direc=str(cache_dir),
    write=True,
)
print('lightcone shape', np.asarray(lc.brightness_temp).shape)

## 4. kSZ — simple coeval-cube method

Integrate $-\sigma_T n_e (v_\parallel/c)$ along the box $z$-axis for each cube.

In [ ]:
ksz_maps = [ksz_from_coeval(c, box_len=BOX_LEN,
                            hlittle=hlittle, OMb=OMb, Y_He=Y_He)
            for c in coevals]

for m in ksz_maps:
    print(f'z={m.redshift:5.2f}  x_HI={m.mean_xHI:.3f}  '
          f'sigma={m.dT_map.std()*1e6:7.2f} uK')

xh = np.array([m.mean_xHI for m in ksz_maps])
mid = int(np.argmin(np.abs(xh - 0.5)))
plot_ksz_map(ksz_maps[mid].dT_map, ksz_maps[mid].BOX_LEN,
             title=f'coeval kSZ  z={ksz_maps[mid].redshift:.2f}  x_HI={ksz_maps[mid].mean_xHI:.2f}',
             out=str(figure_dir / 'ksz_coeval_midpoint.png'))
plt.show()

## 5. kSZ &mdash; full lightcone

Uses the self-contained `ksz_from_lightcone` in `ksz.py`. Runs with and without rotation to compare.

In [ ]:
ksz_rot = ksz_from_lightcone(lc, rotation=True,  z_start=ksz_z_start,
                             parallel_approx=ksz_parallel_approx)
ksz_nor = ksz_from_lightcone(lc, rotation=False, z_start=ksz_z_start,
                             parallel_approx=ksz_parallel_approx)

for name, out in [('rotation=True ', ksz_rot), ('rotation=False', ksz_nor)]:
    print(f'{name}  sigma={out.kSZ_box.std()*1e6:7.2f} uK')

plot_ksz_map(ksz_rot.kSZ_box, BOX_LEN, title='lightcone kSZ (rotation=True)',
             out=str(figure_dir / 'ksz_lightcone_rot.png'))
plt.show()
plot_ksz_map(ksz_nor.kSZ_box, BOX_LEN, title='lightcone kSZ (rotation=False)',
             out=str(figure_dir / 'ksz_lightcone_norot.png'))
plt.show()

fig, ax = plt.subplots(figsize=(7,5))
plot_power_spectrum(ksz_rot.l_s, ksz_rot.kSZ_power, label='rotation=True',
                    xlabel=r'$\ell$', ylabel=r'$\ell(\ell+1) C_\ell / 2\pi$ [$\mu$K$^2$]', ax=ax)
plot_power_spectrum(ksz_nor.l_s, ksz_nor.kSZ_power, label='rotation=False',
                    xlabel=r'$\ell$', ylabel=r'$\ell(\ell+1) C_\ell / 2\pi$ [$\mu$K$^2$]', ax=ax)
fig.savefig(figure_dir / 'ksz_lightcone_Cl.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Non-Gaussian statistics

Mean, variance, skewness, excess kurtosis, 1-point PDF and a 1D power spectrum. Kept inline so you can tweak any of them.

In [ ]:
def field_stats(field_arr, bias_corrected=True):
    """Mean, variance, std, skewness, excess kurtosis of a field."""
    x = np.asarray(field_arr).ravel().astype(np.float64)
    N = x.size
    mu = x.mean()
    y  = x - mu
    var = np.mean(y**2)
    std = np.sqrt(var)
    if std == 0:
        return dict(mean=mu, var=0.0, std=0.0, skewness=0.0, kurtosis=0.0)
    skew = np.mean(y**3) / std**3
    exk  = np.mean(y**4) / var**2 - 3.0
    if bias_corrected and N > 3:
        skew *= np.sqrt(N*(N-1)) / (N-2)
        exk   = ((N-1) / ((N-2)*(N-3))) * ((N+1)*exk + 6)
    return dict(mean=float(mu), var=float(var), std=float(std),
                skewness=float(skew), kurtosis=float(exk))


def field_pdf(field_arr, bins=pdf_bins, rng=None, density=True):
    x = np.asarray(field_arr).ravel()
    pdf, edges = np.histogram(x, bins=bins, range=rng, density=density)
    centers = 0.5 * (edges[:-1] + edges[1:])
    return centers, pdf, edges


def power_spectrum_1d(field_arr, box_size, nbins=pk_nbins,
                      log_bins=pk_log_bins, remove_mean=True):
    """Spherically (3D) / circularly (2D) averaged power spectrum."""
    x = np.asarray(field_arr).astype(np.float64)
    if remove_mean:
        x = x - x.mean()
    ndim, N = x.ndim, x.shape[0]
    if any(s != N for s in x.shape):
        raise ValueError('expects a cubic/square field')
    dx  = box_size / N
    vol = box_size ** ndim
    fk = np.fft.fftn(x) * dx**ndim
    Pk3 = (np.abs(fk)**2) / vol
    kfreq = np.fft.fftfreq(N, d=dx) * 2*np.pi
    if ndim == 2:
        kx, ky = np.meshgrid(kfreq, kfreq, indexing='ij')
        kmag = np.sqrt(kx**2 + ky**2)
    elif ndim == 3:
        kx, ky, kz = np.meshgrid(kfreq, kfreq, kfreq, indexing='ij')
        kmag = np.sqrt(kx**2 + ky**2 + kz**2)
    else:
        raise ValueError('only 2D or 3D')
    kmin = 2*np.pi / box_size
    kmax = np.pi * N / box_size
    edges = (np.logspace(np.log10(kmin), np.log10(kmax), nbins+1)
             if log_bins else np.linspace(kmin, kmax, nbins+1))
    flat_k = kmag.ravel(); flat_P = Pk3.ravel()
    idx = np.digitize(flat_k, edges) - 1
    valid = (idx >= 0) & (idx < nbins) & (flat_k > 0)
    Pk = np.zeros(nbins); counts = np.zeros(nbins, dtype=int)
    for i in range(nbins):
        sel = valid & (idx == i)
        if sel.any():
            Pk[i] = flat_P[sel].mean(); counts[i] = sel.sum()
    centers = 0.5 * (edges[:-1] + edges[1:])
    return centers, Pk, counts

### 6a. Per-coeval stats, evolution vs mean $x_{\rm HI}$

In [ ]:
evo = {'redshift': [], 'mean_xHI': [],
       'mean': [], 'variance': [], 'skewness': [], 'kurtosis': []}

for c, m in zip(coevals, ksz_maps):
    s = field_stats(m.dT_map)
    evo['redshift'].append(float(c.redshift))
    evo['mean_xHI'].append(float(np.mean(c.xH_box)))
    evo['mean'].append(s['mean'])
    evo['variance'].append(s['var'])
    evo['skewness'].append(s['skewness'])
    evo['kurtosis'].append(s['kurtosis'])

# sort by x_HI descending so the EoR progresses left-to-right
order = np.argsort(-np.asarray(evo['mean_xHI']))
evo = {k: np.asarray(v)[order] for k, v in evo.items()}

for i in range(len(evo['mean_xHI'])):
    print(f'z={evo["redshift"][i]:5.2f}  x_HI={evo["mean_xHI"][i]:.3f}  '
          f'var={evo["variance"][i]:.3e}  skew={evo["skewness"][i]:+.3f}  '
          f'kurt={evo["kurtosis"][i]:+.3f}')

plot_evolution(evo, out=str(figure_dir / 'ksz_evolution.png'))
plt.show()

### 6b. PDF evolution

In [ ]:
fields    = [m.dT_map for m in ksz_maps]
xhi_list  = [m.mean_xHI for m in ksz_maps]
z_list    = [m.redshift for m in ksz_maps]
plot_pdf_evolution(fields, xhi_list, z_list, bins=pdf_bins,
                   out=str(figure_dir / 'ksz_pdf_evolution.png'))
plt.show()

### 6c. 2D kSZ power spectrum, per redshift

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))
cmap = plt.get_cmap('viridis')
for j, i in enumerate(order):
    m = ksz_maps[i]
    k, Pk, _ = power_spectrum_1d(m.dT_map, box_size=m.BOX_LEN)
    col = cmap(j / max(len(order)-1, 1))
    ax.loglog(k, np.abs(Pk)*1e12, lw=1.8, color=col,
              label=f'z={m.redshift:.2f}  x_HI={m.mean_xHI:.2f}')
ax.set_xlabel(r'$k$  [Mpc$^{-1}$]')
ax.set_ylabel(r'$P(k)$  [$\mu$K$^2$ Mpc$^2$]')
ax.set_title('coeval 2D kSZ P(k)')
ax.grid(True, which='both', alpha=0.3)
ax.legend(fontsize=8, ncol=2)
fig.savefig(figure_dir / 'ksz_Pk_evolution.png', dpi=150, bbox_inches='tight')
plt.show()

### 6d. Total lightcone-kSZ non-Gaussianity

In [ ]:
total = field_stats(ksz_rot.kSZ_box)
print('total kSZ NG (lightcone, rotation=True):')
for k_, v_ in total.items():
    print(f'  {k_:10s} = {v_:.5e}')

with open(output_dir / 'ksz_lightcone_stats.txt', 'w') as f:
    for k_, v_ in total.items():
        f.write(f'{k_} = {v_}\n')

### 6e. Lightcone kSZ 1-point PDF

Single-map PDF for the rotation-on and rotation-off lightcone kSZ, standardised as $(\Delta T - \langle\Delta T\rangle)/\sigma$ and compared to a unit Gaussian. Any departure from the dashed Gaussian is the kSZ non-Gaussianity.

In [ ]:
# standardised overlay -- compare to unit Gaussian
plot_lightcone_pdf(
    {'rotation=True':  ksz_rot.kSZ_box,
     'rotation=False': ksz_nor.kSZ_box},
    bins=pdf_bins,
    standardise=True,
    title='lightcone kSZ 1-point PDF (standardised)',
    out=str(figure_dir / 'ksz_lightcone_pdf_std.png'),
)
plt.show()


# print summary stats for each
for name, box in [('rotation=True ', ksz_rot.kSZ_box),
                  ('rotation=False', ksz_nor.kSZ_box)]:
    s = field_stats(box)
    print(f'{name}  sigma={s["std"]*1e6:7.2f} uK   '
          f'skew={s["skewness"]:+.3f}   excess_kurt={s["kurtosis"]:+.3f}')

## 7. Save evolution table